# Data Cleaning & Preprocessing

## Objective

This notebook cleans and preprocesses the 9 raw e-commerce datasets based on
the issues identified during the Data Quality Analysis.

The cleaning process follows these principles:

1. Understand the data-quality issue before modifying data.
2. Make cleaning decisions based on business meaning.
3. Avoid blindly dropping nulls, duplicates, or outliers.
4. Preserve valid business information.
5. Validate every transformation after it is applied.
6. Maintain referential integrity across related tables.
7. Save the cleaned datasets separately from the raw datasets.

### Workflow

Data Quality Finding
→ Cleaning Decision
→ Transformation
→ Validation
→ Documentation
→ Cleaned Dataset

In [ ]:
import pandas as pd
import sys
sys.path.insert(0, "../src")
from data_loader import load_datasets

In [ ]:
raw = "../data/raw/"
datasets = load_datasets(raw)

In [ ]:
customers = datasets["customers"]
orders = datasets["orders"]
order_items = datasets["order_items"]
order_payments = datasets["order_payments"]
order_reviews = datasets["order_reviews"]
products = datasets["products"]
sellers = datasets["sellers"]
geolocation = datasets["geolocation"]
translation = datasets["product_category_translation"]

In [ ]:
dataset_shapes = {
    "customers": customers.shape,
    "orders": orders.shape,
    "order_items": order_items.shape,
    "order_payments": order_payments.shape,
    "order_reviews": order_reviews.shape,
    "products": products.shape,
    "sellers": sellers.shape,
    "geolocation": geolocation.shape,
    "translation": translation.shape,
}

dataset_shapes

## 4. Cleaning Strategy

Each dataset will be cleaned independently according to the findings from
the data-quality analysis.

For every important cleaning decision, we will follow:

### Finding
What issue did the data-quality analysis identify?

### Why
Why does this issue matter?

### Decision
What exactly will we change, and why?

### Transformation
Apply the cleaning operation.

### Validation
Confirm that the transformation produced the expected result.

### Documentation
Record the final decision and its business/analytical consequence.

Cleaning will be performed table by table, starting with the simplest tables
and progressing to more complex tables.

### Table Customers

In [ ]:
customers.info()

##  Customer Cleaning

### Findings from Data Quality Analysis

| Check | Finding | Cleaning Action |
|---|---|---|
| Missing values | No missing values | No action |
| Exact duplicates | No exact duplicates | No action |
| Primary key | `customer_id` is unique | No action |
| Customer state | Values appear consistent | No action |
| Customer city | No obvious consistency issue identified | No action |
| ZIP prefix | Stored as integer although it is a geographic code | Convert to string |

### Cleaning Decision

`customer_zip_code_prefix` represents a geographic code rather than a
numeric measurement. Therefore, it should be stored as a string so that
it is treated as an identifier/code rather than a quantity.

In [ ]:
customers["customer_zip_code_prefix"].head()

In [ ]:
customers["customer_zip_code_prefix"] = (
    customers["customer_zip_code_prefix"]
    .astype(str)
)

In [ ]:
customers.dtypes

##  Orders Cleaning

### Findings from Data Quality Analysis

| Check | Finding | Cleaning Action |
|---|---|---|
| Missing values | `order_approved_at`: 160 missing | Investigate by `order_status`; do not blindly fill/drop |
| Missing values | `order_delivered_carrier_date`: 1,783 missing | Mostly expected for incomplete orders; investigate exceptions |
| Missing values | `order_delivered_customer_date`: 2,965 missing | Mostly expected for incomplete orders; investigate exceptions |
| Exact duplicates | No exact duplicate rows | No action |
| Primary key | `order_id` is unique | No action |
| Order status | 8 consistent categories | No action |
| Datatypes | Lifecycle date columns required datetime conversion for analysis | Convert to datetime as part of cleaning |
| Chronological validity | 166 carrier dates occur before purchase | Investigate suspicious records |
| Chronological validity | 23 customer-delivery dates occur before carrier date | Investigate suspicious records |
| Chronological validity | 0 customer-delivery dates occur before purchase | No action |
| Date range | No obviously unrealistic extreme dates identified | No action |

In [ ]:
orders.info()

In [ ]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

for column in date_columns:
    orders[column] = pd.to_datetime(orders[column], errors="coerce")

In [ ]:
orders.info()

In [ ]:
orders.groupby("order_status")[date_columns].apply(
    lambda df: df.isna().sum()
)

In [ ]:
delivered_missing_approved = orders[
    (orders["order_status"] == "delivered") &
    (orders["order_approved_at"].isna())
]

delivered_missing_approved

In [ ]:
delivered_missing_carrier = orders[
    (orders["order_status"] == "delivered") &
    (orders["order_delivered_carrier_date"].isna())
]

delivered_missing_carrier

In [ ]:
delivered_missing_customer_date = orders[
    (orders["order_status"] == "delivered") &
    (orders["order_delivered_customer_date"].isna())
]

delivered_missing_customer_date

### Missing Lifecycle Dates — Cleaning Decision

Missing lifecycle dates were investigated by `order_status`.

Most missing values are consistent with the order lifecycle. For example,
shipped orders may not yet have a customer delivery timestamp, while
canceled or incomplete orders may never reach later delivery stages.

A small number of `delivered` orders have missing lifecycle timestamps:

- 14 missing `order_approved_at`
- 2 missing `order_delivered_carrier_date`
- 8 missing `order_delivered_customer_date`

These timestamps cannot be reliably reconstructed from the available
columns. Therefore, the missing values are retained as `NaT`.

The affected orders are not removed because the orders themselves contain
valid business information.

No arbitrary date imputation is performed because doing so would introduce
unsupported assumptions and could distort downstream delivery metrics.

In [ ]:
orders[
    orders["order_delivered_carrier_date"]
    < orders["order_purchase_timestamp"]
][
    [
        "order_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
    ]
].head(20)

###  Chronological Anomaly — Carrier Date Before Purchase

The data-quality analysis identified 166 orders where
`order_delivered_carrier_date` occurs before `order_purchase_timestamp`.

Inspection of sample records shows that these differences are generally
small time offsets rather than obviously corrupted dates. Several records
also have an approval timestamp occurring after the purchase timestamp,
indicating that the supplied lifecycle timestamps may not always represent
strict event ordering at minute-level precision.

The correct carrier timestamp cannot be reliably reconstructed from the
available data.

Therefore:

- The affected records are retained.
- No carrier timestamps are artificially changed.
- No orders are deleted.
- The anomaly is documented for downstream analysis.

This avoids introducing unsupported assumptions into the dataset.

In [ ]:
orders[
    orders["order_delivered_customer_date"]
    < orders["order_delivered_carrier_date"]
][
    [
        "order_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
    ]
]

### Chronological Anomaly — Delivery Before Carrier

The data-quality analysis identified 23 orders where
`order_delivered_customer_date` occurs before
`order_delivered_carrier_date`.

Inspection shows that these are substantive chronological inconsistencies,
with some differences spanning several days. The correct timestamp cannot
be reliably reconstructed from the available data.

Therefore:

- The affected orders are retained.
- Original timestamps are not artificially modified.
- The affected records are not deleted.
- The anomaly is documented for downstream analysis.

Any delivery metrics that depend on the carrier timestamp should account
for these anomalous records appropriately.

In [ ]:
delivery_before_carrier = orders[
    orders["order_delivered_customer_date"]
    < orders["order_delivered_carrier_date"]
].copy()

(
    delivery_before_carrier["order_delivered_carrier_date"]
    - delivery_before_carrier["order_delivered_customer_date"]
).describe()

### Chronological Anomaly Summary

A total of 23 orders have `order_delivered_customer_date` occurring before
`order_delivered_carrier_date`.

The difference between the two timestamps ranges from approximately
23 minutes to 16 days, with a median of approximately 1.7 days and a mean
of approximately 3.3 days.

These are substantive chronological inconsistencies. However, the correct
timestamp cannot be reliably reconstructed from the available data.

Therefore, the original timestamps are retained and the affected records
are not deleted or artificially corrected.

These records should be considered when creating and interpreting
delivery-related features and analyses.

In [ ]:
orders.shape

In [ ]:
orders["order_id"].is_unique

In [ ]:
orders.isna().sum()

In [ ]:
orders.dtypes

| Check                                   |        Result | Status                    |
| --------------------------------------- | ------------: | ------------------------- |
| Shape                                   | `(99,441, 8)` | ✅ Unchanged               |
| `order_id` uniqueness                   |        `True` | ✅ Preserved               |
| Missing `order_approved_at`             |           160 | ⚠️ Retained intentionally |
| Missing `order_delivered_carrier_date`  |         1,783 | ⚠️ Retained intentionally |
| Missing `order_delivered_customer_date` |         2,965 | ⚠️ Retained intentionally |
| Other columns                           |     0 missing | ✅                         |


##  Order Items Cleaning

###  Findings from Data Quality Analysis

| Check | Finding | Cleaning Action |
|---|---|---|
| Grain | One row represents one item within an order | Preserve this grain |
| Missing values | No missing values | No action |
| Exact duplicates | No exact duplicate rows | No action |
| Composite key | `(order_id, order_item_id)` is unique | No action |
| Foreign keys | `order_id`, `product_id`, and `seller_id` all valid | No action |
| Datatypes | `shipping_limit_date` required datetime conversion | Convert to datetime |
| Negative price | 0 records | No action |
| Zero price | 0 records | No action |
| Negative freight | 0 records | No action |
| Zero freight | 383 records | Retain; not proven invalid |
| Extreme price/freight values | Inspected and appear plausible | Retain |
| Shipping limit before purchase | 0 records | No action |

In [ ]:
order_items.info()

In [ ]:
order_items["shipping_limit_date"] = pd.to_datetime(
    order_items["shipping_limit_date"],
    errors="coerce"
)

In [ ]:
order_items.dtypes

In [ ]:
order_items["shipping_limit_date"].isna().sum()

In [ ]:
(order_items["freight_value"] == 0).sum()

In [ ]:
order_items.shape

In [ ]:
order_items[["order_id", "order_item_id"]].drop_duplicates().shape

In [ ]:
order_items.isna().sum()

In [ ]:
(order_items["freight_value"] == 0).sum()

### Order Items Cleaning — Final Conclusion

The `order_items` table was successfully cleaned and validated.

#### Changes applied

- Converted `shipping_limit_date` to datetime.
- Retained all zero `freight_value` records because zero freight may represent
  legitimate business conditions such as free shipping.
- Retained extreme price and freight values because they were inspected and
  not determined to be invalid.

#### Validation

- Row count remained 112,650.
- Composite key `(order_id, order_item_id)` remains unique.
- No missing values were introduced.
- All 383 zero-freight records remain in the dataset.
- No rows were deleted.

The table is considered cleaned without removing potentially valid
transaction information.

##  Order Payments Cleaning

### Findings from Data Quality Analysis

| Check | Finding | Cleaning Action |
|---|---|---|
| Grain | One row represents one payment record for an order | Preserve this grain |
| Missing values | No missing values | No action |
| Exact duplicates | No exact duplicate rows | No action |
| Composite key | `(order_id, payment_sequential)` is unique | No action |
| Foreign key | `order_id` values are valid | No action |
| Datatypes | Columns are broadly appropriate | Review/standardize if needed |
| Payment type | 5 categories including `not_defined` | Retain; no unsupported category removal |
| Payment installments | 2 records have `0` installments | Investigate before deciding |
| Payment value | No negative values | No action |
| Payment value | 9 records have `0` value | Investigate before deciding |
| Extreme payment value | Maximum value is 13,664.08 and was inspected as plausible | Retain |
| Payment rows per order | Most orders have one payment; some have multiple | Preserve one-row-per-payment grain |
| Delivered orders without payment | 1 delivered order has no payment record | Investigate; do not alter blindly |

In [ ]:
order_payments.info()

In [ ]:
order_payments[
    order_payments["payment_installments"] == 0
]

### Zero Payment Installments

The data-quality analysis identified 2 payment records where
`payment_installments` is 0.

Both records have `payment_type = credit_card` and positive
`payment_value`.

The available data does not provide enough information to reliably
determine the intended number of installments.

Therefore:

- The two records are retained.
- `payment_installments = 0` is documented as an anomalous value.
- No value is imputed.
- No payment records are deleted.

In [ ]:
order_payments[
    order_payments["payment_value"] == 0
]

### Zero Payment Values

The data-quality analysis identified 9 payment records where
`payment_value = 0`.

Inspection showed that these records belong to the `voucher` and
`not_defined` payment types. All have `payment_installments = 1`, and
several occur as payment records within orders having multiple payment
sequences.

The available data does not provide sufficient evidence to classify these
zero-value records as invalid.

Therefore:

- Zero payment values are retained.
- No values are imputed or replaced.
- No payment records are deleted.

These records will be preserved so that payment-level information is not
artificially altered.

In [ ]:
delivered_orders_without_payment = orders[
    (orders["order_status"] == "delivered") &
    (~orders["order_id"].isin(order_payments["order_id"]))
]

delivered_orders_without_payment

### Delivered Order Without Payment Record

One `delivered` order was identified without a corresponding record in
`order_payments`.

The order contains a complete purchase-to-delivery lifecycle, but the
supplied payment dataset does not contain a matching payment record.

The correct payment method and payment amount cannot be reliably
reconstructed from the available data.

Therefore:

- The order is retained.
- No synthetic payment record is created.
- No payment value is imputed.
- The order is documented as having missing payment information.

Downstream payment-based analyses should account for this unmatched order.

In [ ]:
order_payments.shape

In [ ]:
order_payments[["order_id", "payment_sequential"]].drop_duplicates().shape

In [ ]:
order_payments.isna().sum()

In [ ]:
(order_payments["payment_installments"] == 0).sum()

In [ ]:
(order_payments["payment_value"] == 0).sum()

### Order Payments Cleaning — Final Conclusion

The `order_payments` table was cleaned and validated.

#### Cleaning decisions

- No missing values required treatment.
- Composite key `(order_id, payment_sequential)` was preserved.
- The 2 records with `payment_installments = 0` were retained because the
  intended installment count cannot be reliably determined.
- The 9 records with `payment_value = 0` were retained because they occur
  with `voucher` or `not_defined` payment types and cannot be established as
  invalid from the available data.
- Extreme payment values were retained after inspection because they appear
  plausible.
- The one delivered order without a matching payment record was retained and
  documented rather than creating a synthetic payment record.

No payment records were deleted or artificially modified.

## Order Reviews Cleaning

### Findings from Data Quality Analysis

| Check | Finding | Cleaning Action |
|---|---|---|
| Grain | One row represents a review record associated with an order | Preserve this grain |
| Missing values | `review_comment_title`: 87,656 missing | Retain; optional text |
| Missing values | `review_comment_message`: 58,247 missing | Retain; optional text |
| Exact duplicates | 0 | No action |
| `review_id` uniqueness | 814 duplicate occurrences; 789 duplicated review IDs | Investigate key/grain before deciding |
| `(review_id, order_id)` | Unique | Preserve as a valid composite identifier |
| Foreign key | `order_id` values valid | No action |
| Review score | Values only 1–5 | No action |
| Date columns | Review dates required datetime conversion | Convert to datetime |
| Answer vs creation date | No answers before review creation | No action |
| Review creation vs purchase | 74 reviews created before purchase | Investigate |
| Review text missingness | Substantial but optional | Do not blindly fill |

In [ ]:
order_reviews.info()

In [ ]:
order_reviews[
    order_reviews["review_id"].duplicated(keep=False)
].sort_values("review_id").head(20)

###  Review ID Duplication — Cleaning Decision

The data-quality analysis identified 814 duplicate occurrences of
`review_id`.

Inspection shows that duplicated `review_id` values can be associated with
different `order_id` values while sharing the same review score, review text,
creation date, and answer timestamp.

Therefore, `review_id` alone should not be treated as a unique identifier in
the supplied dataset.

The combination `(review_id, order_id)` is unique and is therefore retained
as the appropriate composite identifier for review records.

The duplicated `review_id` values are not removed because doing so could
discard valid order-review relationships.

No rows are deleted as a result of this finding.

In [ ]:
review_date_columns = [
    "review_creation_date",
    "review_answer_timestamp",
]

for column in review_date_columns:
    order_reviews[column] = pd.to_datetime(
        order_reviews[column],
        errors="coerce"
    )

In [ ]:
order_reviews.info()

In [ ]:
order_reviews[review_date_columns].isna().sum()


In [ ]:
reviews_before_purchase = order_reviews.merge(
    orders[
        [
            "order_id",
            "order_purchase_timestamp",
            "order_status"
        ]
    ],
    on="order_id",
    how="left"
)

reviews_before_purchase = reviews_before_purchase[
    reviews_before_purchase["review_creation_date"]
    < reviews_before_purchase["order_purchase_timestamp"]
]

reviews_before_purchase[
    [
        "review_id",
        "order_id",
        "order_status",
        "review_score",
        "review_creation_date",
        "order_purchase_timestamp",
        "review_answer_timestamp"
    ]
].head(20)

### Review Creation Before Purchase

A total of 74 review records have `review_creation_date` earlier than the
corresponding `order_purchase_timestamp`.

Inspection shows that most of these records are associated with canceled
orders, and some have substantial differences between the recorded review
creation date and purchase timestamp.

The available data does not provide enough information to determine which
timestamp is incorrect or to reliably reconstruct the intended value.

Therefore:

- The original timestamps are retained.
- No review dates are artificially changed.
- No review rows are deleted.
- The 74 records are documented as chronological anomalies.

These anomalies should be considered when interpreting any analysis involving
review timing.

In [ ]:
order_reviews[["review_comment_title", "review_comment_message"]].isna().sum()

In [ ]:
(
    order_reviews["review_comment_title"].isna()
    & order_reviews["review_comment_message"].isna()
).sum()

In [ ]:
order_reviews.shape

In [ ]:
order_reviews[["review_id", "order_id"]].drop_duplicates().shape

In [ ]:
order_reviews.isna().sum()

In [ ]:
order_reviews["review_score"].between(1, 5).all()

In [ ]:
order_reviews.info()

###  Order Reviews Cleaning — Final Conclusion

The `order_reviews` table was cleaned and validated.

#### Cleaning decisions

- Converted `review_creation_date` and `review_answer_timestamp` to datetime.
- Retained duplicated `review_id` values because `review_id` alone is not
  unique in the supplied data.
- Preserved `(review_id, order_id)` as the appropriate unique combination.
- Retained missing review titles and messages because review text is optional.
- Retained the 74 review records with creation dates before purchase because
  the correct timestamp could not be reliably reconstructed.
- No review records were deleted.
- No review text or timestamps were artificially created.

#### Validation

- Table shape remained unchanged.
- `(review_id, order_id)` remains unique.
- No unexpected missing values were introduced.
- `review_score` remains within the valid 1–5 range.

The table is considered cleaned while preserving the limitations present in
the supplied source data.

##  Products Cleaning

### Findings from Data Quality Analysis

| Check | Finding | Cleaning Action |
|---|---|---|
| Grain | One row represents one product | Preserve this grain |
| Missing values | 610 records have missing category/descriptive attributes | Investigate before treatment |
| Missing values | 2 records have missing physical attributes | Investigate before treatment |
| Exact duplicates | No exact duplicate rows | No action |
| Primary key | `product_id` is unique | No action |
| Datatypes | Numeric attributes are `float64` because of missing values | Review after missing-value decision |
| Category values | No obvious categorical inconsistency | No action |
| Negative values | No negative numeric values | No action |
| Zero values | No zero numeric values | No action |
| Extreme weight | Maximum weight is 40,425 g; inspected values appear plausible | Retain |
| Dimensions | Inspected and appear plausible | Retain |

In [ ]:
products.info()

In [ ]:
products.isna().sum()

In [ ]:
products[products.isna().any(axis=1)].head(10)

### Missing Product Attributes — 610 Records

The data-quality analysis identified 610 products with missing values in:

- `product_category_name`
- `product_name_lenght`
- `product_description_lenght`
- `product_photos_qty`

Inspection shows that these missing values occur together, while physical
attributes such as weight and dimensions may still be present.

These attributes cannot be reliably reconstructed from the available product
data. Imputing categories or descriptive attributes would introduce
unsupported assumptions.

Therefore:

- The 610 records are retained.
- The missing values remain `NaN`.
- No category or descriptive values are artificially imputed.
- No products are deleted because of these missing attributes.

Downstream analyses that require these attributes should explicitly handle
the missing values.

In [ ]:
products[
    products[
        [
            "product_weight_g",
            "product_length_cm",
            "product_height_cm",
            "product_width_cm",
        ]
    ].isna().any(axis=1)
]

In [ ]:
products.shape

In [ ]:
products["product_id"].is_unique

In [ ]:
products.isna().sum()

In [ ]:
(products.select_dtypes(include="number") < 0).sum()

### Products Cleaning — Final Conclusion

The `products` table was cleaned and validated.

#### Cleaning decisions

- Retained the 610 products with missing category/descriptive attributes
  because the missing values could not be reliably reconstructed.
- Retained the 2 products with missing physical attributes because their
  correct measurements could not be determined from the available data.
- Retained extreme but plausible weight and dimension values.
- No product records were deleted.
- No missing product attributes were artificially imputed.

#### Validation

- Row count remained unchanged.
- `product_id` remains unique.
- Missing values remain limited to the source-data cases identified during
  data-quality analysis.
- No negative numeric product measurements were introduced.

The table is considered cleaned while preserving valid product records and
avoiding unsupported assumptions.

##  Seller Cleaning

###  Findings from Data Quality Analysis

| Check | Finding | Cleaning Action |
|---|---|---|
| Grain | One row represents one seller | Preserve this grain |
| Missing values | No missing values | No action |
| Exact duplicates | No exact duplicate rows | No action |
| Primary key | `seller_id` is unique | No action |
| Seller state | 23 uppercase state codes; appeared consistent | No action |
| Seller city | 611 distinct cities; no obvious issue in initial inspection | No action |
| Seller ZIP prefix | Stored as integer geographic code | Review representation/type |
| ZIP prefix range | 1,001–99,730 | No obvious invalid value identified |
| ZIP prefix → state | 17 ZIP prefixes map to multiple states | Investigate before changing |
| Multi-state ZIP records | Often involve different city/state combinations | Do not automatically modify |
| Geolocation relationship | Geolocation data may help interpret ambiguous ZIPs | Consider during enrichment/analysis |

In [ ]:
sellers["seller_zip_code_prefix"].dtype

In [ ]:
sellers["seller_zip_code_prefix"].head(10)

In [ ]:
sellers["seller_zip_code_prefix"] = (
    sellers["seller_zip_code_prefix"].astype(str)
)

In [ ]:
sellers["seller_zip_code_prefix"].dtype

In [ ]:
sellers["seller_zip_code_prefix"].str.len().value_counts().sort_index()

In [ ]:
seller_zip_state_variants = (
    sellers.groupby("seller_zip_code_prefix")["seller_state"]
    .unique()
)

seller_zip_state_variants[
    seller_zip_state_variants.str.len() > 1
]

In [ ]:
sellers[
    sellers["seller_zip_code_prefix"].isin(
        seller_zip_state_variants[
            seller_zip_state_variants.str.len() > 1
        ].index
    )
].sort_values("seller_zip_code_prefix")

In [ ]:
problem_zips = seller_zip_state_variants[
    seller_zip_state_variants.str.len() > 1
].index

geolocation[
    geolocation["geolocation_zip_code_prefix"].isin(problem_zips)
][
    [
        "geolocation_zip_code_prefix",
        "geolocation_city",
        "geolocation_state"
    ]
].drop_duplicates().sort_values(
    ["geolocation_zip_code_prefix", "geolocation_state", "geolocation_city"]
)

In [ ]:
geolocation["geolocation_zip_code_prefix"].dtype

In [ ]:
sellers["seller_zip_code_prefix"].dtype

In [ ]:
geolocation_zip = geolocation["geolocation_zip_code_prefix"].astype(str)

In [ ]:
geolocation[
    geolocation_zip.isin(problem_zips)
][
    [
        "geolocation_zip_code_prefix",
        "geolocation_city",
        "geolocation_state"
    ]
].drop_duplicates().sort_values(
    ["geolocation_zip_code_prefix", "geolocation_state", "geolocation_city"]
)

In [ ]:
geo_problem_zips = (
    geolocation[
        geolocation["geolocation_zip_code_prefix"].astype(str).isin(problem_zips)
    ]
    .groupby("geolocation_zip_code_prefix")["geolocation_state"]
    .unique()
)

geo_problem_zips

In [ ]:
geo_state_map = (
    geolocation[
        geolocation["geolocation_zip_code_prefix"].astype(str).isin(problem_zips)
    ]
    .groupby("geolocation_zip_code_prefix")["geolocation_state"]
    .first()
)

seller_check = sellers[
    sellers["seller_zip_code_prefix"].isin(problem_zips)
].copy()

seller_check["geo_state"] = (
    seller_check["seller_zip_code_prefix"]
    .map(geo_state_map)
)

seller_check[
    seller_check["seller_state"] != seller_check["geo_state"]
][
    [
        "seller_id",
        "seller_zip_code_prefix",
        "seller_city",
        "seller_state",
        "geo_state"
    ]
]

In [ ]:
geo_state_map = (
    geolocation.assign(
        zip_prefix_str=geolocation["geolocation_zip_code_prefix"].astype(str)
    )
    [
        geolocation["geolocation_zip_code_prefix"].astype(str).isin(problem_zips)
    ]
    .groupby("zip_prefix_str")["geolocation_state"]
    .first()
)

seller_check = sellers[
    sellers["seller_zip_code_prefix"].isin(problem_zips)
].copy()

seller_check["geo_state"] = (
    seller_check["seller_zip_code_prefix"]
    .map(geo_state_map)
)

seller_check[
    [
        "seller_id",
        "seller_zip_code_prefix",
        "seller_city",
        "seller_state",
        "geo_state"
    ]
]

In [ ]:
seller_state_map = geo_state_map.to_dict()

mismatch_mask = (
    sellers["seller_zip_code_prefix"].isin(problem_zips)
    & sellers["seller_state"].ne(
        sellers["seller_zip_code_prefix"].map(seller_state_map)
    )
)

sellers.loc[mismatch_mask, "seller_state"] = (
    sellers.loc[mismatch_mask, "seller_zip_code_prefix"]
    .map(seller_state_map)
)

In [ ]:
(
    sellers["seller_zip_code_prefix"].isin(problem_zips)
    & sellers["seller_state"].ne(
        sellers["seller_zip_code_prefix"].map(seller_state_map)
    )
).sum()

In [ ]:
sellers["seller_id"].is_unique

### Seller Cleaning — Final Conclusion

The `sellers` table was cleaned and validated.

#### Cleaning decisions

- Converted `seller_zip_code_prefix` from integer to string because it is a
  geographic code rather than a numeric measurement.
- Preserved the original ZIP-prefix values without zero-padding.
- Investigated the 17 ZIP prefixes associated with multiple seller states.
- Used the `geolocation` table as a project-internal reference after
  confirming that each of those 17 ZIP prefixes mapped to one unique state.
- Corrected the 19 seller records whose `seller_state` conflicted with the
  validated geolocation state.
- Seller records with matching states were left unchanged.

#### Validation

- No seller records were deleted.
- `seller_id` remains unique.
- Table shape remains unchanged.
- All identified seller ZIP/state mismatches were resolved.

The table is considered cleaned while preserving the original seller records
and using validated project data to resolve the identified state
inconsistencies.

##  Geolocation Cleaning

### Findings from Data Quality Analysis

| Check | Finding | Cleaning Action |
|---|---|---|
| Grain | One row represents one geolocation observation | Preserve this grain |
| Missing values | No missing values | No action |
| Exact duplicates | 261,831 exact duplicate rows (~26.18%) | Investigate and deduplicate if justified |
| ZIP prefix uniqueness | ZIP prefix is not unique; 981,148 repeated occurrences | Expected structural behavior; do not deduplicate by ZIP |
| Latitude validity | No values outside `[-90, 90]` | No action |
| Longitude validity | No values outside `[-180, 180]` | No action |
| State values | 27 two-letter uppercase state codes | No action |
| City cardinality | 8,011 distinct cities | No action by itself |
| City naming variation | 2,085 distinct city values contain accented characters | Standardize if justified |
| Same ZIP + state → city | Multiple city labels observed for some ZIP/state combinations | Review as text-standardization issue |

In [ ]:
geolocation.info()

In [ ]:
geolocation.shape

In [ ]:
geolocation.duplicated().sum()

In [ ]:
geolocation.drop_duplicates().shape

### Exact Duplicate Rows — Cleaning Decision

The `geolocation` table contains 261,831 exact duplicate rows.

The table's grain is one geolocation observation. An exact duplicate has
identical ZIP prefix, latitude, longitude, city, and state values and
therefore provides no additional geographic information.

Keeping exact duplicates could also unintentionally give repeated
observations greater influence in downstream geographic enrichment or
aggregation.

Therefore, exact duplicate rows are removed.

This does not mean that repeated ZIP prefixes are removed. Multiple
non-identical observations for the same ZIP prefix are retained because
they may represent different geographic observations.

In [ ]:
geolocation = geolocation.drop_duplicates().reset_index(drop=True)

In [ ]:
geolocation.shape

In [ ]:
geolocation.duplicated().sum()

In [ ]:
geolocation["geolocation_zip_code_prefix"].nunique()


In [ ]:
geolocation["geolocation_zip_code_prefix"].is_unique

In [ ]:
city_normalized = (
    geolocation["geolocation_city"]
    .str.normalize("NFKD")
    .str.encode("ascii", errors="ignore")
    .str.decode("utf-8")
)

geolocation["geolocation_city"].nunique(), city_normalized.nunique()

In [ ]:
(city_normalized != geolocation["geolocation_city"]).sum()

In [ ]:
geolocation.loc[
    city_normalized != geolocation["geolocation_city"],
    ["geolocation_city"]
].drop_duplicates().head(30)

In [ ]:
city_normalized = (
    geolocation["geolocation_city"]
    .str.normalize("NFKD")
    .str.encode("ascii", errors="ignore")
    .str.decode("utf-8")
)

(
    geolocation.groupby(city_normalized)["geolocation_city"]
    .nunique()
    .sort_values(ascending=False)
    .head(20)
)

In [ ]:
geolocation.loc[
    city_normalized != geolocation["geolocation_city"],
    ["geolocation_city"]
].drop_duplicates().head(100)

### City Name Standardization — Basic Normalization

The `geolocation_city` column contains textual variations such as accented
and unaccented spellings. For example, `são paulo` and `sao paulo` represent
the same city but would otherwise be treated as different categorical values.

The investigation also identified other forms of variation, including missing
spaces and potentially corrupted characters. These require additional
evidence before correction.

Therefore, the first standardization step is limited to:

- removing Unicode diacritics,
- trimming leading/trailing whitespace,
- collapsing repeated whitespace.

More complex corrections such as missing spaces, punctuation changes, or
encoding corruption will not be applied automatically without sufficient
evidence.

In [ ]:
geolocation["geolocation_city"] = (
    geolocation["geolocation_city"]
    .str.normalize("NFKD")
    .str.encode("ascii", errors="ignore")
    .str.decode("utf-8")
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

In [ ]:
geolocation["geolocation_city"].nunique()

In [ ]:
geolocation[
    geolocation["geolocation_city"].isin(
        ["sao paulo", "saopaulo", "sa£o paulo"]
    )
]["geolocation_city"].value_counts()

In [ ]:
geolocation[
    geolocation["geolocation_city"] == "saopaulo"
][
    [
        "geolocation_zip_code_prefix",
        "geolocation_city",
        "geolocation_state",
        "geolocation_lat",
        "geolocation_lng"
    ]
]

In [ ]:
geolocation.loc[
    geolocation["geolocation_city"] == "saopaulo",
    "geolocation_city"
] = "sao paulo"

In [ ]:
geolocation["geolocation_city"].isin(
    ["saopaulo", "sao paulo"]
).value_counts()

In [ ]:
geolocation["geolocation_city"].nunique()

### Specific City Formatting Correction

After basic text normalization, 2 records remained with the city value
`saopaulo`.

Inspection showed that both records have state `SP` and geographic
coordinates consistent with the other `sao paulo` records. The difference
is therefore a formatting variation caused by a missing space.

These two records were standardized from `saopaulo` to `sao paulo`.

No broad spacing rule was applied to all city names because such a rule could
incorrectly modify legitimate city names.

In [ ]:
geolocation[
    geolocation["geolocation_city"].str.contains(
        r"[^a-zA-Z0-9\s\-\.]",
        regex=True,
        na=False
    )
]["geolocation_city"].drop_duplicates().head(100)

In [ ]:
special_city_values = (
    geolocation[
        geolocation["geolocation_city"].str.contains(
            r"[^a-zA-Z0-9\s\-\.]",
            regex=True,
            na=False
        )
    ]["geolocation_city"]
    .drop_duplicates()
    .sort_values()
)

special_city_values.to_list()

In [ ]:
from urllib.parse import unquote
from html import unescape

city_decoded = (
    geolocation["geolocation_city"]
    .map(unquote)
    .map(unescape)
)

In [ ]:
city_decoded = (
    city_decoded
    .str.normalize("NFKD")
    .str.encode("ascii", errors="ignore")
    .str.decode("utf-8")
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

In [ ]:
encoding_changes = (
    geolocation.loc[
        city_decoded != geolocation["geolocation_city"],
        ["geolocation_city"]
    ]
    .drop_duplicates()
    .copy()
)

encoding_changes["normalized_city"] = (
    encoding_changes["geolocation_city"].map(
        lambda x: (
            unquote(x)
        )
    ).map(unescape)
)

encoding_changes

In [ ]:
encoding_changes = (
    pd.DataFrame({
        "original": geolocation["geolocation_city"],
        "normalized": city_decoded
    })
    .drop_duplicates()
)

encoding_changes[
    encoding_changes["original"] != encoding_changes["normalized"]
].head(100)

In [ ]:
geolocation["geolocation_city"] = city_decoded

In [ ]:
geolocation[
    geolocation["geolocation_city"].isin([
        "sao joao do pau d%26apos%3balho",
        "lambari d%26apos%3boeste",
        "florian&oacute;polis"
    ])
]["geolocation_city"].value_counts()

In [ ]:
geolocation["geolocation_city"].nunique()


In [ ]:
geolocation[
    geolocation["geolocation_city"].str.contains(
        r"%|&\w+;",
        regex=True,
        na=False
    )
]["geolocation_city"].drop_duplicates().head(50)

In [ ]:
geolocation[
    geolocation["geolocation_city"] == "* cidade"
]

In [ ]:
geolocation[
    geolocation["geolocation_city"] == "* cidade"
][
    [
        "geolocation_zip_code_prefix",
        "geolocation_city",
        "geolocation_state",
        "geolocation_lat",
        "geolocation_lng"
    ]
]

In [ ]:
geolocation["geolocation_zip_code_prefix"].nunique()

In [ ]:
(geolocation["geolocation_lat"].abs() > 90).sum(), (
    geolocation["geolocation_lng"].abs() > 180
).sum()

###  Geolocation Cleaning — Final Conclusion

The `geolocation` table was cleaned and validated.

#### Cleaning decisions

- Removed 261,831 exact duplicate rows.
- Preserved repeated ZIP prefixes because multiple observations per ZIP
  are part of the table's grain.
- Standardized city names by removing diacritics and normalizing whitespace.
- Corrected the 2 `saopaulo` records to `sao paulo` based on geographic
  context.
- Decoded the 3 verified URL/HTML-encoded city values.
- Preserved legitimate punctuation and locality descriptions.
- Retained the single unresolved `* cidade` label because its correct city
  could not be reliably determined from the supplied data.

#### Validation

- Final shape: 738,332 rows × 5 columns.
- Exact duplicate rows: 0.
- Missing values: 0.
- Unique ZIP prefixes: 19,015.
- Invalid latitude values: 0.
- Invalid longitude values: 0.

The table is considered cleaned while preserving valid geographic
observations and avoiding unsupported corrections.

## Product Category Translation Cleaning

###  Findings from Data Quality Analysis

| Check | Finding | Cleaning Action |
|---|---|---|
| Grain | One row represents one product category translation | Preserve this grain |
| Missing values | No missing values | No action |
| Exact duplicates | No exact duplicate rows | No action |
| `product_category_name` uniqueness | All 71 values are unique | No action |
| `product_category_name_english` uniqueness | All 71 values are unique | No action |
| Category mapping | 71-to-71 one-to-one mapping | Preserve mapping |

In [ ]:
translation.info()

In [ ]:
translation.isna().sum()

In [ ]:
translation.duplicated().sum()

In [ ]:
translation["product_category_name"].is_unique

In [ ]:
translation["product_category_name_english"].is_unique

##  Cross-Table Validation

After cleaning the individual datasets, cross-table relationships are
validated again.

The objective is to confirm that:

- primary/composite keys remain valid,
- foreign-key relationships remain intact,
- no cleaning operation accidentally created orphan records,
- expected table grains are preserved.

Because the project contains several one-to-many relationships, these
checks validate relationships without performing unnecessary joins.

In [ ]:
orders["customer_id"].isin(customers["customer_id"]).all()

In [ ]:
order_items["order_id"].isin(orders["order_id"]).all()

In [ ]:
order_items["product_id"].isin(products["product_id"]).all()

In [ ]:
order_items["seller_id"].isin(sellers["seller_id"]).all()

In [ ]:
order_payments["order_id"].isin(orders["order_id"]).all()

In [ ]:
order_reviews["order_id"].isin(orders["order_id"]).all()

In [ ]:
translation["product_category_name"].isin(
    products["product_category_name"].dropna()
).all()

### Cross-Table Validation — Conclusion

All validated foreign-key relationships remain intact after cleaning.

| Relationship | Validation |
|---|---|
| `orders.customer_id` → `customers.customer_id` | ✅ Valid |
| `order_items.order_id` → `orders.order_id` | ✅ Valid |
| `order_items.product_id` → `products.product_id` | ✅ Valid |
| `order_items.seller_id` → `sellers.seller_id` | ✅ Valid |
| `order_payments.order_id` → `orders.order_id` | ✅ Valid |
| `order_reviews.order_id` → `orders.order_id` | ✅ Valid |
| `translation.product_category_name` → `products.product_category_name` | ✅ Valid |

No orphan foreign-key records were introduced by the cleaning process.
The relationships required for downstream SQL analysis, feature engineering,
and EDA remain intact.

In [ ]:
final_shapes = {
    "customers": customers.shape,
    "orders": orders.shape,
    "order_items": order_items.shape,
    "order_payments": order_payments.shape,
    "order_reviews": order_reviews.shape,
    "products": products.shape,
    "sellers": sellers.shape,
    "geolocation": geolocation.shape,
    "translation": translation.shape,
}

final_shapes

In [ ]:
assert customers["customer_id"].is_unique
assert orders["order_id"].is_unique
assert products["product_id"].is_unique
assert sellers["seller_id"].is_unique

assert order_items[["order_id", "order_item_id"]].drop_duplicates().shape[0] == len(order_items)
assert order_payments[["order_id", "payment_sequential"]].drop_duplicates().shape[0] == len(order_payments)
assert order_reviews[["review_id", "order_id"]].drop_duplicates().shape[0] == len(order_reviews)

## 16. Save Cleaned Datasets

The cleaned datasets are saved separately from the raw datasets in
`data/cleaned/`.

The raw files are preserved unchanged so that the cleaning process remains
reproducible.

In [ ]:
from pathlib import Path

project_root = Path.cwd().parent
cleaned_dir = project_root / "data" / "cleaned"

cleaned_dir.mkdir(parents=True, exist_ok=True)

cleaned_dir

In [ ]:
cleaned_datasets = {
    "customers": customers,
    "orders": orders,
    "order_items": order_items,
    "order_payments": order_payments,
    "order_reviews": order_reviews,
    "products": products,
    "sellers": sellers,
    "geolocation": geolocation,
    "product_category_name_translation": translation,
}

for name, df in cleaned_datasets.items():
    df.to_csv(cleaned_dir / f"{name}.csv", index=False)

In [ ]:
sorted(path.name for path in cleaned_dir.glob("*.csv"))

In [ ]:
Path.cwd()

## Notebook Conclusion

The 9 raw datasets were systematically cleaned based on the findings from
the data-quality analysis.

Cleaning decisions were made using business meaning and evidence from the
supplied data. Values were changed only when the transformation could be
justified; otherwise, the original information was preserved and the
limitation was documented.

The cleaned datasets were validated for structure, key integrity,
referential integrity, and important data-quality conditions, and were
saved separately under `data/cleaned/`.

**Status: Data Cleaning & Preprocessing — COMPLETE ✅**

In [135]:
from data_loader import load_cleaned_datasets
cleaned = "../data/cleaned/"
clean_datasets = load_cleaned_datasets(cleaned)

In [136]:
for name, df in clean_datasets.items():
    print(f"\n{"--"*15} {name} {"--"*15}")
    print(df.info())


------------------------------ customers ------------------------------
<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  int64
 3   customer_city             99441 non-null  str  
 4   customer_state            99441 non-null  str  
dtypes: int64(1), str(4)
memory usage: 11.0 MB
None

------------------------------ orders ------------------------------
<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   ord